In [1]:
import torch
from argparse import Namespace
from ay2.tools.text._phonemes import Phonemer_Tokenizer_Recombination
from pandas import Series

torch.serialization.add_safe_globals([
    Namespace,
    Phonemer_Tokenizer_Recombination,
    Series,
])

ModuleNotFoundError: No module named 'librosa'

In [2]:
import torch

_orig_torch_load = torch.load

def _torch_load_no_weights_only(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_torch_load(*args, **kwargs)

torch.load = _torch_load_no_weights_only
print("✅ Patched torch.load to default weights_only=False for this kernel session.")

✅ Patched torch.load to default weights_only=False for this kernel session.


In [3]:
import os, sys
notebook_dir = os.getcwd()
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

In [4]:
from phoneme_GAT.phoneme_model import BaseModule, load_phoneme_model, optim_param
from ay2.tools.text._phonemes import Phonemer_Tokenizer_Recombination
from phoneme_GAT.modules import Phoneme_GAT_lit, Phoneme_GAT

/lambda/nfs/algo/workspace/DeepfakeDetectionRenewed/venv/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [5]:
import os

def load_hf_token(path="secret.txt"):
    if os.path.exists(path):
        with open(path, "r") as f:
            return f.read().strip()
    return None

DATASET_NAME = "Bisher/ASVspoof_2019_LA"
CACHE_DIR    = "./data/asvspoof_2019_la"
HF_TOKEN     = load_hf_token()

print("Dataset :", DATASET_NAME)
print("HF token:", "✅ Yes" if HF_TOKEN else "❌ No")

Dataset : Bisher/ASVspoof_2019_LA
HF token: ✅ Yes


## Model config

In [6]:
from argparse import Namespace

cfg = Namespace(
    PhonemeGAT=Namespace(
        backbone="wavlm",
        use_raw=False,
        use_GAT=True,
        n_edges=10,              # 10 like Copy notebook
        use_aug=True,
        use_pool=True,
        use_clip=True,
    )
)

In [7]:
import math, random, torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset, Audio as HFAudio, concatenate_datasets
from loader import _make_balanced_indices, _label_to_int, _crop_policy, TARGET_SR
from collections import Counter
import torchaudio.functional as F_audio
from fractions import Fraction

# ── Load and concatenate train + test splits ──────────────────────────────
print("Loading HF train + test splits...")
hf_train = load_dataset(DATASET_NAME, split="train", cache_dir=CACHE_DIR,
                         token=HF_TOKEN if HF_TOKEN else None)
hf_test  = load_dataset(DATASET_NAME, split="test",  cache_dir=CACHE_DIR,
                         token=HF_TOKEN if HF_TOKEN else None)
hf_train = hf_train.cast_column("audio", HFAudio(sampling_rate=TARGET_SR))
hf_test  = hf_test.cast_column("audio",  HFAudio(sampling_rate=TARGET_SR))

hf_pool = concatenate_datasets([hf_train, hf_test])
print(f"Combined pool size: {len(hf_pool)}")

_label_key = next(
    (k for k in hf_pool[0].keys() if "label" in k.lower() or k.lower() == "key"),
    "label"
)
print(f"Label key: '{_label_key}'")

# ── Balanced 50/50 index list, limit=30000 (15k bonafide + 15k spoof) ────
_labels  = [_label_to_int(hf_pool[i][_label_key]) for i in range(len(hf_pool))]
_indices = _make_balanced_indices(_labels, seed=42, limit=30000)

_counts = Counter(_label_to_int(hf_pool[i][_label_key]) for i in _indices)
print(f"Balanced pool: {len(_indices)} samples ({_counts[0]} bonafide + {_counts[1]} spoof)")

/lambda/nfs/algo/workspace/DeepfakeDetectionRenewed/loader.py:18: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("sox_io")


Loading HF train + test splits...
Combined pool size: 96617
Label key: 'key'
Balanced pool: 19870 samples (9935 bonafide + 9935 spoof)


In [8]:
# ── Augmentation functions ────────────────────────────────────────────────
def _match_len(x, n):
    if x.shape[-1] < n:
        x = torch.nn.functional.pad(x, (0, n - x.shape[-1]))
    return x[..., :n]

def aug_noise(wav):
    snr_db = random.uniform(8, 40)
    sig_power = wav.pow(2).mean()
    noise = torch.randn_like(wav)
    noise_power = noise.pow(2).mean()
    scale = (sig_power / (noise_power * 10 ** (snr_db / 10))).sqrt()
    return wav + scale * noise

def aug_pitch_up(wav):
    semitones = random.uniform(1, 5)
    factor = 2 ** (semitones / 12)
    frac = Fraction(factor).limit_denominator(20)
    return _match_len(F_audio.resample(wav, frac.numerator, frac.denominator), wav.shape[-1])

def aug_reverb(wav):
    t60 = random.choice([0.20, 0.35, 0.50, 0.70, 0.90])
    room_scale = random.uniform(0.25, 0.80)
    T = wav.shape[-1]
    rir_len = min(int(t60 * TARGET_SR), 1600)
    t = torch.linspace(0, t60, rir_len)
    decay = torch.exp(-6.9 * t / t60)
    rir = torch.randn(rir_len) * decay
    for d_ms in [15, 30, 50]:
        d = int(d_ms * 1e-3 * TARGET_SR * room_scale)
        if d < rir_len:
            rir[d] += 0.4 * room_scale * decay[d]
    rir = rir / (rir.abs().max() + 1e-8)
    n_fft = 2 ** math.ceil(math.log2(T + rir_len - 1))
    out = torch.fft.irfft(torch.fft.rfft(wav, n=n_fft) * torch.fft.rfft(rir, n=n_fft), n=n_fft)
    return out[..., :T]

# ── Proportional aug sampling weights (derived from goat.pth sweep) ───────
# struggle = 1 - mean_val_acc:  noise=0.221, pitch_up=0.206, reverb=0.156
# normalised to sum=1 within augmented portion:
#   noise=50%  pitch_up=30%  reverb=20%
AUG_FNS      = [aug_noise,  aug_pitch_up, aug_reverb]
AUG_WEIGHTS  = [0.50,       0.30,         0.20]
AUG_NAMES    = ["noise",    "pitch_up",   "reverb"]
AUG_PROB     = 0.35   # 35% of samples get augmented → 65% stay clean


class RobustDataset(Dataset):
    """
    30k balanced samples (15k bonafide + 15k spoof).
    65% clean, 35% augmented.
    Within augmented: noise 50%, pitch_up 30%, reverb 20%
    — proportional to goat.pth struggle scores from the attack sweep.
    """
    def __init__(self, hf_ds, indices, label_key, aug_fns, aug_weights, aug_prob):
        self.ds          = hf_ds
        self.indices     = indices
        self.label_key   = label_key
        self.aug_fns     = aug_fns
        self.aug_weights = aug_weights
        self.aug_prob    = aug_prob

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        ex  = self.ds[self.indices[idx]]
        wav = torch.tensor(ex["audio"]["array"], dtype=torch.float32).unsqueeze(0)
        wav = _crop_policy(wav, "train")
        if random.random() < self.aug_prob:
            aug_fn = random.choices(self.aug_fns, weights=self.aug_weights, k=1)[0]
            wav = aug_fn(wav)
        y = _label_to_int(ex[self.label_key])
        return {"audio": wav, "label": torch.tensor(y).long(), "sample_rate": TARGET_SR}


train_dataset = RobustDataset(
    hf_pool, _indices, _label_key,
    AUG_FNS, AUG_WEIGHTS, AUG_PROB,
)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True,
)

b = next(iter(train_dataloader))
print("Train batch audio :", b["audio"].shape)
print("Train batch labels:", b["label"].shape)
print(f"Dataset: {len(train_dataset)} samples | {len(train_dataloader)} batches/epoch")
print(f"Aug breakdown (of 35% augmented): noise={AUG_WEIGHTS[0]*100:.0f}%  pitch_up={AUG_WEIGHTS[1]*100:.0f}%  reverb={AUG_WEIGHTS[2]*100:.0f}%")

Train batch audio : torch.Size([20, 1, 48000])
Train batch labels: torch.Size([20])
Dataset: 19870 samples | 993 batches/epoch
Aug breakdown (of 35% augmented): noise=50%  pitch_up=30%  reverb=20%


In [9]:
from loader import get_eval_dataloader

val_dataloader = get_eval_dataloader(
    source="hf",
    split="validation",
    hf_name=DATASET_NAME,
    batch_size=20,
    hf_cache_dir=CACHE_DIR,
    limit=3000,
    hf_token=HF_TOKEN if HF_TOKEN else None,
)
print(f"Val samples: {len(val_dataloader.dataset)}")

✓ HF Bisher/ASVspoof_2019_LA:validation [eval] → 3000 samples (balance=True)
Val samples: 3000


In [10]:
from callbacks_rational import (
    BinaryACC_Callback, BinaryAUC_Callback, EER_Callback,
    TPR_Callback, TNR_Callback, FPR_Callback, FNR_Callback,
)

## Model

In [11]:
audio_model_lit = Phoneme_GAT_lit(cfg=cfg)

Now, load vocab json files from /lambda/nfs/algo/workspace/DeepfakeDetectionRenewed/vocab_phoneme Please make sure the vocab files are correct
Load WavLM model!!!!!!!


Some weights of WavLMForCTC were not initialized from the model checkpoint at microsoft/wavlm-base and are newly initialized: ['encoder.pos_conv_embed.conv.parametrizations.weight.original1', 'encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'lm_head.weight', 'lm_head.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


torch.Size([687, 768])


## Train

In [12]:
import wandb
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import WandbLogger

if wandb.run is not None:
    wandb.finish()

wandb_logger = WandbLogger(
    project="DeepfakeDetectionRenewed",
    entity="krishrawat0222-f",
    name="robust-goat-train",
    log_model=True,
    tags=["robust", "multi-aug", "30k", "from-scratch"],
)
wandb_logger.experiment.config.update({
    "train_samples":   30000,
    "aug_prob":        0.35,
    "clean_prob":      0.65,
    "aug_weights":     dict(zip(AUG_NAMES, AUG_WEIGHTS)),
    "balance":         "50/50 bonafide/spoof",
    "n_edges":         10,
    "max_epochs":      7,
    "batch_size":      20,
    "aug_rationale":   "proportional to goat.pth struggle scores from attack sweep",
}, allow_val_change=True)

trainer = Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=7,
    logger=wandb_logger,
    callbacks=[
        BinaryACC_Callback(batch_key="label", output_key="logit"),
        BinaryAUC_Callback(batch_key="label", output_key="logit"),
        EER_Callback(batch_key="label", output_key="logit"),
        TPR_Callback(batch_key="label", output_key="logit"),
        TNR_Callback(batch_key="label", output_key="logit"),
        FPR_Callback(batch_key="label", output_key="logit"),
        FNR_Callback(batch_key="label", output_key="logit"),
    ],
    log_every_n_steps=10,
)

wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/ubuntu/.netrc.
wandb: Currently logged in as: yashaspatil (krishrawat0222-f) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [13]:
trainer.fit(audio_model_lit, train_dataloader, val_dataloader)

You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type                    | Params | Mode 
------------------------------------------------------------------
0 | model         | Phoneme_GAT             | 106 M  | train
1 | bce_loss      | BCEWithLogitsLoss       | 0      | train
2 | ce_loss       | CrossEntropyLoss        | 0      | train
3 | contrast_loss | BinaryTokenContrastLoss | 0      | train
4 | clip_head     | Sequential              | 1.2 M  | train
5 | clip_loss     | CLIPLoss1D              | 1      | train
------------------------------------------------------------------
12.9 M    Trainable params
94.9 M 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=7` reached.


In [14]:
torch.save(audio_model_lit.state_dict(), "robust_goat.pth")
trainer.save_checkpoint("robust_goat.ckpt")
print("✅ Saved robust_goat.pth + robust_goat.ckpt")

if wandb.run is not None:
    wandb.finish()

✅ Saved robust_goat.pth + robust_goat.ckpt


epoch,▁▁▂▂▃▃▅▅▆▆▇▇██
train-acc,▁▃▅▆▇██
train-auc,▁▄▅▆▇██
train-aug_loss,█▃▂▂▁▁▁
train-clip_loss,█▃▂▂▁▁▁
train-cls_loss,█▆▅▃▂▂▁
train-eer,█▆▄▂▂▁▁
train-fnr,█▆▄▃▂▁▁
train-fpr,█▆▅▃▂▂▁
train-loss,█▄▃▂▂▁▁
+14,...
